In [1]:
import os

# Ensure all relative paths (gw/, lookup/, reports/) resolve from the project root,
# regardless of whether the notebook is opened from notebooks/ or notebooks/transcripts_processor/.
cwd = os.getcwd()
if os.path.basename(cwd) == "text_processor":
    os.chdir("../..")

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/augustustranter/Documents/Programming/Projects/fpl/fpl_agent


In [2]:
# This variable indicates the gameweek for which we are generating a report.
# E.g. if gameweek = 3, transcripts in gw/gw3/transcripts/ are analysed.
gameweek = 5

# 1. FILTER TEXT

In [4]:
import importlib.util
import sys
from pathlib import Path

filter_text_path = Path("notebooks/text_processor/1_filter_text.py")
filter_spec = importlib.util.spec_from_file_location("text_filter", filter_text_path)
assert filter_spec is not None and filter_spec.loader is not None
filter_module = importlib.util.module_from_spec(filter_spec)
sys.modules[filter_spec.name] = filter_module
filter_spec.loader.exec_module(filter_module)
process_text_folder = filter_module.process_text_folder

text_cache_dir = Path(f"gw/gw{gameweek}/text_cache")
filter_cache_dir = text_cache_dir / "filter_cache"
segment_words = 10
max_workers = 4

filtered_text_sources = process_text_folder(
    gameweek=gameweek,
    cache_dir=filter_cache_dir,
    segment_words=segment_words,
    max_workers=max_workers,
    refresh=False,
)

filtered_text_payload = {
    source_name: filtered.text
    for source_name, filtered in filtered_text_sources.items()
}

print(f"Filter cache written to: {filter_cache_dir}")
print(f"Loaded {len(filtered_text_sources)} filtered text source(s) into memory")
print(f"Segment target word count: {segment_words}")
print(f"Parallel workers: {max_workers}")
print(sorted(filtered_text_payload))

Filter cache written to: gw/gw5/text_cache/filter_cache
Loaded 25 filtered text source(s) into memory
Segment target word count: 10
Parallel workers: 4
['4718882', '8ywgdwco', 'best_fpl_gameweek_5_team_2026_27', 'c69qr50nynqdo', 'fpl_challenge_expert_reveals_their_gameweek_5_team', 'fpl_gameweek_5_captaincy_2026', 'fpl_gameweek_5_differential_picks_103855031_html', 'fpl_gameweek_5_questions_26_27_article', 'fpl_gameweek_5_team_news_fridays_live_injury_updates_3', 'fpl_gameweek_5_transfer_tips_2026_27', 'fpl_gameweek_5_which_teams_are_projected_to_score_the_most_goals', 'fpl_gameweek_5_wildcard_elite_manager_drafts', 'fpl_gw5_fixtures_and_players_to_target_for_gameweek_5', 'fpl_gw5_tips_everything_you_need_to_know_before_the_deadline', 'fpl_gw5_ultimate_guide_and_tips', 'fpl_long_view_who_to_buy_and_sell_based_on_the_upcoming_fixtures', 'latest_premier_league_player_injuries_club_by_club_news', 'scout_selection_best_fantasy_team_for_gameweek_5', 'the_scout_squad_our_top_picks_for_fpl_ga

# 2. MORE FILTERING

In [5]:
import importlib.util
import sys
from pathlib import Path

selection_text_path = Path("notebooks/text_processor/2_selection_filter.py")
selection_spec = importlib.util.spec_from_file_location("text_selection", selection_text_path)
assert selection_spec is not None and selection_spec.loader is not None
selection_module = importlib.util.module_from_spec(selection_spec)
sys.modules[selection_spec.name] = selection_module
selection_spec.loader.exec_module(selection_module)
select_text_sources = selection_module.select_text_sources

text_selection_cache_dir = text_cache_dir / "selection_cache"
selection_model = "gpt-4.1-mini"
selection_max_workers = 4

selected_text_sources = select_text_sources(
    filtered_text_sources,
    cache_dir=text_selection_cache_dir,
    model=selection_model,
    max_workers=selection_max_workers,
    refresh=False,
)

selected_text_payload = {
    source_name: selected.text
    for source_name, selected in selected_text_sources.items()
}

print(f"Selection cache written to: {text_selection_cache_dir}")
print(f"Loaded {len(selected_text_sources)} selected text source(s) into memory")
print(f"Selection model: {selection_model}")
print(f"Parallel workers: {selection_max_workers}")
print(sorted(selected_text_payload))

Selection cache written to: gw/gw5/text_cache/selection_cache
Loaded 25 selected text source(s) into memory
Selection model: gpt-4.1-mini
Parallel workers: 4
['4718882', '8ywgdwco', 'best_fpl_gameweek_5_team_2026_27', 'c69qr50nynqdo', 'fpl_challenge_expert_reveals_their_gameweek_5_team', 'fpl_gameweek_5_captaincy_2026', 'fpl_gameweek_5_differential_picks_103855031_html', 'fpl_gameweek_5_questions_26_27_article', 'fpl_gameweek_5_team_news_fridays_live_injury_updates_3', 'fpl_gameweek_5_transfer_tips_2026_27', 'fpl_gameweek_5_which_teams_are_projected_to_score_the_most_goals', 'fpl_gameweek_5_wildcard_elite_manager_drafts', 'fpl_gw5_fixtures_and_players_to_target_for_gameweek_5', 'fpl_gw5_tips_everything_you_need_to_know_before_the_deadline', 'fpl_gw5_ultimate_guide_and_tips', 'fpl_long_view_who_to_buy_and_sell_based_on_the_upcoming_fixtures', 'latest_premier_league_player_injuries_club_by_club_news', 'scout_selection_best_fantasy_team_for_gameweek_5', 'the_scout_squad_our_top_picks_for_

# 3. EXTRACT TEXT

In [6]:
import importlib.util
import json
from pathlib import Path

extract_text_path = Path("notebooks/text_processor/3_extract_text.py")
extract_spec = importlib.util.spec_from_file_location("text_extract", extract_text_path)
assert extract_spec is not None and extract_spec.loader is not None
extract_module = importlib.util.module_from_spec(extract_spec)
extract_spec.loader.exec_module(extract_module)
build_combined_output = extract_module.build_combined_output
process_filtered_text_sources = extract_module.process_filtered_text_sources

text_extraction_cache_dir = text_cache_dir / "extraction_cache"
text_extraction_output_path = Path(f"gw/gw{gameweek}/data/gw_{gameweek}_text_extractions.json")
extraction_model = "gpt-4.1-mini"
extraction_max_workers = 4
extraction_batch_size = 4

text_extraction_results = process_filtered_text_sources(
    selected_text_payload,
    cache_dir=text_extraction_cache_dir,
    model=extraction_model,
    max_workers=extraction_max_workers,
    batch_size=extraction_batch_size,
    refresh=False,
)

combined_text_extraction = build_combined_output(gameweek, text_extraction_results)
text_extraction_output_path.parent.mkdir(parents=True, exist_ok=True)
text_extraction_output_path.write_text(
    json.dumps(combined_text_extraction, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Extraction cache written to: {text_extraction_cache_dir}")
print(f"Combined extraction JSON written to: {text_extraction_output_path}")
print(f"Extracted {len(combined_text_extraction['recommendations'])} recommendation(s)")
print(f"Processed {len(text_extraction_results)} source(s) in batches of {extraction_batch_size}")

Extraction cache written to: gw/gw5/text_cache/extraction_cache
Combined extraction JSON written to: gw/gw5/data/gw_5_text_extractions.json
Extracted 114 recommendation(s)
Processed 24 source(s) in batches of 4
